# Module 9: Invariant Causal Prediction (ICP) for Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Implement ICP-based feature selection using time-period environments
2. Compare ICP-selected features with standard ML selection (Lasso, SHAP)
3. Test both feature sets under distribution shift (held-out future period)
4. Demonstrate that causal features maintain performance while predictive features degrade

## Prerequisites

- Module 9 Guide 02: Invariant Causal Prediction
- Notebook 01: Causal Discovery and Markov Blanket
- Understanding of linear regression and F-tests

## Estimated Time: 15 minutes

## Key Reference
Peters, J., Bühlmann, P., & Meinshausen, N. (2016). Causal inference by using invariant prediction. *JRSS-B*, 78(5), 947–1012.

## 1. Setup and Data

In [ ]:
# Purpose: Load libraries and prepare a time-series dataset
# Key concept: ICP requires multiple genuine environments — time periods work well

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.multitest import multipletests
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("Setup complete.")

## 2. Synthetic Dataset with Known Causal Structure

We construct a dataset where we know the true causal structure. This lets us verify that ICP correctly identifies causal features.

**Data-generating process:**
- True causal features: $X_1, X_2$ (stable coefficients across environments)
- Spurious features: $X_3, X_4$ (correlated with $Y$ only in some environments)
- Noise features: $X_5, X_6$ (no relationship with $Y$)

**Environments:** 4 time periods. In each period, the distribution of $X_3, X_4$ shifts relative to $Y$, breaking their spurious correlations.

In [ ]:
# Purpose: Generate dataset with known causal structure across multiple environments
# Key concept: causal features have stable coefficients, spurious features do not

def generate_causal_dataset(n_per_env: int = 400,
                             n_envs: int = 4,
                             noise_std: float = 0.5,
                             seed: int = 42):
    """
    Generate a dataset with known causal and spurious features.

    True causal model: Y = 2*X1 + 1.5*X2 + noise

    X3, X4 are confounded by hidden variable H that varies across environments.
    In some environments H correlates positively with Y (via X3, X4),
    in others negatively — making them environment-specific spurious correlates.

    Parameters
    ----------
    n_per_env : int
        Samples per environment.
    n_envs : int
        Number of environments (time periods).
    noise_std : float
        Standard deviation of outcome noise.
    seed : int
        Random seed.

    Returns
    -------
    X : np.ndarray, shape (n_per_env*n_envs, 6)
    y : np.ndarray, shape (n_per_env*n_envs,)
    env_labels : np.ndarray, shape (n_per_env*n_envs,)
    """
    rng = np.random.RandomState(seed)
    n_total = n_per_env * n_envs

    X = np.zeros((n_total, 6))
    y = np.zeros(n_total)
    env_labels = np.zeros(n_total, dtype=int)

    # Environment-specific hidden variable strengths
    # Sign alternates, creating spurious sign reversal in X3, X4
    env_hidden_strength = [2.0, -1.5, 1.0, -2.0]

    for e in range(n_envs):
        start = e * n_per_env
        end = start + n_per_env

        # Causal features: stable across environments
        X[start:end, 0] = rng.randn(n_per_env)  # X1: causal
        X[start:end, 1] = rng.randn(n_per_env)  # X2: causal

        # Hidden confounder: strength varies across environments
        H = env_hidden_strength[e] * rng.randn(n_per_env)

        # X3, X4: driven by hidden confounder (spurious correlates)
        X[start:end, 2] = H + 0.5 * rng.randn(n_per_env)  # X3
        X[start:end, 3] = 0.8 * H + 0.3 * rng.randn(n_per_env)  # X4

        # X5, X6: pure noise features
        X[start:end, 4] = rng.randn(n_per_env)  # X5: noise
        X[start:end, 5] = rng.randn(n_per_env)  # X6: noise

        # Outcome: causal model + H (confounder in Y) + noise
        y[start:end] = (2.0 * X[start:end, 0] +     # True causal coefficient
                        1.5 * X[start:end, 1] +     # True causal coefficient
                        0.3 * H +                    # Hidden confounder also affects Y
                        noise_std * rng.randn(n_per_env))

        env_labels[start:end] = e

    return X, y, env_labels

feature_names = ['X1_causal', 'X2_causal', 'X3_spurious', 'X4_spurious', 'X5_noise', 'X6_noise']
TRUE_CAUSAL = ['X1_causal', 'X2_causal']

X, y, env_labels = generate_causal_dataset(n_per_env=400, n_envs=4)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(env_labels))} environments")
print(f"True causal features: {TRUE_CAUSAL}")
print(f"Spurious features: ['X3_spurious', 'X4_spurious']")
print(f"Noise features: ['X5_noise', 'X6_noise']")

# Show how correlations with Y vary across environments
print("\nCorrelation with Y by environment (spurious features should vary):")
header = f"{'Feature':15s}" + "".join([f"  Env {e}" for e in range(4)])
print(header)
for j, fname in enumerate(feature_names):
    row = f"{fname:15s}"
    for e in range(4):
        mask = env_labels == e
        corr = np.corrcoef(X[mask, j], y[mask])[0, 1]
        row += f"  {corr:+.3f}"
    print(row)

## 3. The Invariance Test

For each candidate feature set $S$, we test whether the regression residuals from $Y \sim X_S$ are invariant across environments. The test has two stages:
1. **Coefficient stability:** F-test comparing pooled vs per-environment regression
2. **Residual variance equality:** Levene's test

In [ ]:
# Purpose: Implement the two-stage invariance test from Peters et al. (2016)
# Key concept: test whether residuals from Y ~ X_S are the same in all environments

def test_invariance(y: np.ndarray,
                    X_S: np.ndarray,
                    env_labels: np.ndarray,
                    alpha: float = 0.05) -> dict:
    """
    Test H0(S): residuals from Y ~ X_S are invariant across environments.

    Two-stage test:
    1. F-test for coefficient equality across environments
    2. Levene's test for residual variance equality

    Reject H0(S) if either test rejects (Bonferroni).

    Parameters
    ----------
    y : array, shape (n,)
        Target variable.
    X_S : array, shape (n, |S|)
        Feature matrix for candidate set S.
    env_labels : array, shape (n,)
        Integer environment labels.
    alpha : float
        Significance level.

    Returns
    -------
    dict with keys: 'invariant' (bool), 'p_value' (float),
                    'p_coef' (Stage 1 p-value), 'p_var' (Stage 2 p-value)
    """
    envs = np.unique(env_labels)
    E = len(envs)
    n = len(y)

    # Handle empty feature set
    if X_S.shape[1] == 0:
        return {'invariant': False, 'p_value': 0.0, 'p_coef': 0.0, 'p_var': 0.0}

    p_s = X_S.shape[1]
    residuals_per_env = []

    # Fit separate model per environment
    for e in envs:
        mask = env_labels == e
        if mask.sum() <= p_s + 1:
            return {'invariant': True, 'p_value': 1.0, 'p_coef': 1.0, 'p_var': 1.0}
        model_e = LinearRegression(fit_intercept=True)
        model_e.fit(X_S[mask], y[mask])
        residuals_per_env.append(y[mask] - model_e.predict(X_S[mask]))

    # Stage 1: F-test for coefficient equality
    # Compare pooled regression vs per-environment regressions
    pooled_model = LinearRegression(fit_intercept=True)
    pooled_model.fit(X_S, y)
    rss_pooled = np.sum((y - pooled_model.predict(X_S)) ** 2)
    rss_separate = sum(np.sum(r ** 2) for r in residuals_per_env)

    df1 = p_s * (E - 1)          # Numerator degrees of freedom
    df2 = n - E * (p_s + 1)      # Denominator degrees of freedom

    if df2 <= 0 or rss_separate < 1e-10:
        p_coef = 1.0
    else:
        f_stat = ((rss_pooled - rss_separate) / df1) / (rss_separate / df2)
        f_stat = max(f_stat, 0.0)  # Guard against numerical negatives
        p_coef = 1.0 - stats.f.cdf(f_stat, df1, df2)

    # Stage 2: Levene's test for residual variance equality
    if len(residuals_per_env) >= 2 and all(len(r) > 1 for r in residuals_per_env):
        p_var = stats.levene(*residuals_per_env).pvalue
    else:
        p_var = 1.0

    # Combined: Bonferroni correction (two tests)
    p_combined = min(min(p_coef, p_var) * 2, 1.0)

    return {
        'invariant': p_combined > alpha,
        'p_value': p_combined,
        'p_coef': p_coef,
        'p_var': p_var,
    }

# Quick verification: the true causal set should pass invariance
causal_idx = [0, 1]  # X1_causal, X2_causal
spurious_idx = [2, 3]  # X3_spurious, X4_spurious

result_causal = test_invariance(y, X[:, causal_idx], env_labels)
result_spurious = test_invariance(y, X[:, spurious_idx], env_labels)
result_all = test_invariance(y, X, env_labels)

print("Invariance test results (alpha=0.05):")
print(f"  True causal features (X1, X2):       "
      f"invariant={result_causal['invariant']}, p={result_causal['p_value']:.4f}")
print(f"  Spurious features (X3, X4):           "
      f"invariant={result_spurious['invariant']}, p={result_spurious['p_value']:.4f}")
print(f"  All features:                         "
      f"invariant={result_all['invariant']}, p={result_all['p_value']:.4f}")
print()
print("Expected: True causal features PASS (invariant=True)")
print("Expected: Spurious features FAIL (invariant=False)")

## 4. ICP: Greedy Implementation

Exhaustive ICP (testing all $2^p$ subsets) is intractable for $p > 20$. We use greedy forward ICP:
at each step, add the feature that maintains invariance and most reduces residual variance.

In [ ]:
# Purpose: Implement greedy ICP for tractable causal feature selection
# Key concept: forward selection using invariance test as the stopping criterion

def greedy_icp(y: np.ndarray,
               X: np.ndarray,
               env_labels: np.ndarray,
               feature_names: list,
               alpha: float = 0.05,
               max_features: int = 15,
               verbose: bool = True) -> dict:
    """
    Greedy forward ICP: select features that maintain invariance.

    At each step:
    1. Test each remaining feature added to current selected set
    2. Among those that pass invariance, add the one that most reduces RSS
    3. Stop when no remaining feature can be added while maintaining invariance

    Parameters
    ----------
    y : array
        Target variable.
    X : array
        Full feature matrix.
    env_labels : array
        Environment labels.
    feature_names : list
        Feature names.
    alpha : float
        Significance level for invariance test.
    max_features : int
        Maximum features to select.
    verbose : bool
        Print selection steps.

    Returns
    -------
    dict with 'selected_idx', 'selected_names', 'steps'
    """
    p = X.shape[1]
    selected_idx = []
    remaining_idx = list(range(p))
    steps = []

    if verbose:
        print(f"Starting greedy ICP (alpha={alpha})...")

    for step in range(max_features):
        best_feature = None
        best_rss = np.inf
        best_p_value = None

        for j in remaining_idx:
            candidate_idx = selected_idx + [j]
            X_cand = X[:, candidate_idx]
            result = test_invariance(y, X_cand, env_labels, alpha=alpha)

            if result['invariant']:
                # Among invariant candidates, pick the one minimising pooled RSS
                pooled_model = LinearRegression(fit_intercept=True)
                pooled_model.fit(X_cand, y)
                rss = np.sum((y - pooled_model.predict(X_cand)) ** 2)
                if rss < best_rss:
                    best_rss = rss
                    best_feature = j
                    best_p_value = result['p_value']

        if best_feature is None:
            if verbose:
                print(f"  Step {step+1}: No more features maintain invariance. Stopping.")
            break

        selected_idx.append(best_feature)
        remaining_idx.remove(best_feature)
        steps.append({'step': step+1, 'feature': feature_names[best_feature],
                      'p_value': best_p_value, 'rss': best_rss})

        if verbose:
            print(f"  Step {step+1}: Add '{feature_names[best_feature]}' "
                  f"(invariance p={best_p_value:.4f}, RSS={best_rss:.2f})")

    selected_names = [feature_names[i] for i in selected_idx]
    return {
        'selected_idx': selected_idx,
        'selected_names': selected_names,
        'steps': steps,
    }

# Standardise data before ICP
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

icp_result = greedy_icp(y, X_scaled, env_labels, feature_names, alpha=0.05)

In [ ]:
# Purpose: Summarise ICP results and compare with ground truth
# Key concept: ICP should select causal features and exclude spurious ones

print("=" * 55)
print("ICP Feature Selection Results")
print("=" * 55)
print(f"\nICP selected: {icp_result['selected_names']}")
print(f"True causal:  {TRUE_CAUSAL}")

icp_set = set(icp_result['selected_names'])
true_set = set(TRUE_CAUSAL)

true_positives = icp_set & true_set
false_positives = icp_set - true_set
false_negatives = true_set - icp_set

print(f"\nTrue positives (correctly selected causal): {true_positives}")
print(f"False positives (spurious features included): {false_positives}")
print(f"False negatives (causal features missed): {false_negatives}")

precision = len(true_positives) / len(icp_set) if icp_set else 0
recall = len(true_positives) / len(true_set) if true_set else 0
print(f"\nPrecision (among selected, fraction truly causal): {precision:.2f}")
print(f"Recall (among causal, fraction selected): {recall:.2f}")

## 5. Standard Feature Selection for Comparison

In [ ]:
# Purpose: Run Lasso and SHAP-based selection for comparison
# Key concept: standard methods optimise predictive accuracy, not causal correctness

# Use environments 0-2 as training (past), environment 3 as test (future — distribution shift)
train_mask = env_labels < 3
test_mask = env_labels == 3

X_train = X_scaled[train_mask]
y_train = y[train_mask]
X_test = X_scaled[test_mask]
y_test = y[test_mask]

print(f"Training: {train_mask.sum()} samples (environments 0-2)")
print(f"Test (shift): {test_mask.sum()} samples (environment 3, held-out future period)")

# ── Lasso ──────────────────────────────────────────────────────────────────
lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train, y_train)
lasso_features = [feature_names[i] for i in np.where(np.abs(lasso.coef_) > 1e-4)[0]]
print(f"\nLasso selected: {lasso_features}")

# ── SHAP (Gradient Boosting) ───────────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
explainer = shap.TreeExplainer(gb)
shap_values = explainer.shap_values(X_train)
shap_imp = np.abs(shap_values).mean(axis=0)

# Select features with SHAP importance above median
shap_threshold = np.median(shap_imp)
shap_features = [feature_names[i] for i in np.where(shap_imp > shap_threshold)[0]]
print(f"SHAP selected: {shap_features}")
print(f"ICP selected:  {icp_result['selected_names']}")
print(f"True causal:   {TRUE_CAUSAL}")

## 6. Distribution Shift Test

Environment 3 has the most extreme spurious correlation (hidden strength = -2.0). Features $X_3, X_4$ will have reversed correlations with $Y$ compared to environments 0–2. We test:
- **ICP features:** Should maintain performance (causally stable)
- **Lasso/SHAP features:** May degrade (include spurious features)

In [ ]:
# Purpose: Evaluate each feature set on in-distribution vs shifted test set
# Key concept: causal features should degrade less under shift

def evaluate_feature_set_split(feat_names_subset: list,
                                X_train: np.ndarray, y_train: np.ndarray,
                                X_test: np.ndarray, y_test: np.ndarray,
                                all_feature_names: list) -> dict:
    """
    Train GradientBoosting on train set, evaluate on test (shifted) set.
    Also evaluate on a 20% held-out split of training set (in-distribution).
    """
    if not feat_names_subset:
        return {'train_r2': np.nan, 'test_r2': np.nan}

    indices = [all_feature_names.index(f) for f in feat_names_subset
               if f in all_feature_names]
    if not indices:
        return {'train_r2': np.nan, 'test_r2': np.nan}

    X_tr = X_train[:, indices]
    X_te = X_test[:, indices]

    model = GradientBoostingRegressor(
        n_estimators=150, max_depth=3, learning_rate=0.1, random_state=42
    )
    model.fit(X_tr, y_train)

    train_r2 = r2_score(y_train, model.predict(X_tr))
    test_r2 = r2_score(y_test, model.predict(X_te))

    return {'train_r2': train_r2, 'test_r2': test_r2}

all_feat_names = feature_names

# Evaluate each feature set
evaluation_sets = {
    'ICP (causal)': icp_result['selected_names'],
    'Lasso': lasso_features,
    'SHAP': shap_features,
    'True causal': TRUE_CAUSAL,
    'All features': all_feat_names,
}

eval_results = {}
print(f"{'Method':20s}  {'Features':35s}  {'Train R²':10s}  {'Test R² (shift)':15s}  {'Degradation'}")
print("-" * 100)
for method, feat_list in evaluation_sets.items():
    result = evaluate_feature_set_split(
        feat_list, X_train, y_train, X_test, y_test, all_feat_names
    )
    eval_results[method] = result
    degrad = result['train_r2'] - result['test_r2'] if not np.isnan(result['train_r2']) else np.nan
    feat_str = str(feat_list)[:35]
    print(f"{method:20s}  {feat_str:35s}  {result['train_r2']:10.4f}  "
          f"{result['test_r2']:15.4f}  {degrad:+.4f}")

In [ ]:
# Purpose: Visualise performance under distribution shift
# Key concept: causal features show smaller degradation than predictive features

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = list(eval_results.keys())
train_r2s = [eval_results[m]['train_r2'] for m in methods]
test_r2s = [eval_results[m]['test_r2'] for m in methods]
degradations = [t - s for t, s in zip(train_r2s, test_r2s)]

colors = ['#4a90d9', '#e74c3c', '#27ae60', '#9b59b6', '#95a5a6']

# Plot 1: Train vs Test R2
x = np.arange(len(methods))
width = 0.35
axes[0].bar(x - width/2, train_r2s, width, label='Train R² (in-distribution)',
            color=colors, alpha=0.9)
axes[0].bar(x + width/2, test_r2s, width, label='Test R² (distribution shift)',
            color=colors, alpha=0.5, hatch='//')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, fontsize=9, rotation=15)
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].set_title('In-distribution vs Distribution Shift Performance', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Performance degradation (lower = more robust)
bar_colors = ['#2ecc71' if d < 0.1 else '#e74c3c' for d in degradations]
axes[1].bar(x, degradations, color=bar_colors, alpha=0.85)
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, fontsize=9, rotation=15)
axes[1].set_ylabel('R² Degradation (Train - Test)', fontsize=11)
axes[1].set_title('Distribution Shift Robustness\n(lower = more robust)', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# Annotate bars with values
for bar, val in zip(axes[1].patches, degradations):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'{val:+.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey finding:")
icp_degrad = eval_results['ICP (causal)']['train_r2'] - eval_results['ICP (causal)']['test_r2']
lasso_degrad = eval_results['Lasso']['train_r2'] - eval_results['Lasso']['test_r2']
print(f"  ICP degradation:   {icp_degrad:+.4f}")
print(f"  Lasso degradation: {lasso_degrad:+.4f}")
print(f"  ICP is {'more' if icp_degrad < lasso_degrad else 'less'} robust to distribution shift.")

## 7. Visualising Invariance Across Environments

We visualise why causal features pass the invariance test and spurious features do not.

In [ ]:
# Purpose: Visualise the invariance property — stable coefficients for causal features
# Key concept: invariant conditional distributions are the signal ICP uses

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
env_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
env_names = ['Env 0 (train)', 'Env 1 (train)', 'Env 2 (train)', 'Env 3 (test/shift)']

features_to_plot = [
    ('X1_causal', 0, 'Causal: stable slope across environments'),
    ('X2_causal', 1, 'Causal: stable slope across environments'),
    ('X3_spurious', 2, 'Spurious: slope reverses across environments'),
    ('X4_spurious', 3, 'Spurious: slope reverses across environments'),
    ('X5_noise', 4, 'Noise: no systematic relationship'),
    ('X6_noise', 5, 'Noise: no systematic relationship'),
]

for ax, (feat_name, feat_idx, title) in zip(axes.flat, features_to_plot):
    for e in range(4):
        mask = env_labels == e
        x_e = X_scaled[mask, feat_idx]
        y_e = y[mask]

        # Fit environment-specific regression
        slope = np.cov(x_e, y_e)[0, 1] / np.var(x_e)
        intercept = y_e.mean() - slope * x_e.mean()

        # Scatter (subsample for clarity)
        sample_idx = np.random.choice(len(x_e), size=min(80, len(x_e)), replace=False)
        ax.scatter(x_e[sample_idx], y_e[sample_idx],
                   alpha=0.2, s=10, color=env_colors[e])

        # Regression line
        x_range = np.linspace(x_e.min(), x_e.max(), 50)
        ax.plot(x_range, intercept + slope * x_range,
                color=env_colors[e], linewidth=2,
                label=f'{env_names[e]} (slope={slope:.2f})')

    ax.set_xlabel(feat_name, fontsize=10)
    ax.set_ylabel('Y', fontsize=10)
    ax.set_title(title, fontsize=9, wrap=True)
    ax.legend(fontsize=7, loc='upper left')

plt.suptitle('Per-Environment Regression Slopes: Causal vs Spurious Features',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Causal features (X1, X2): regression slopes are STABLE across environments")
print("- Spurious features (X3, X4): regression slopes REVERSE sign — invariance fails")
print("- ICP detects this instability and excludes X3, X4")

## 8. Exercise: ICP with More Environments

**Task:** Generate a new dataset with 6 environments (instead of 4) and re-run ICP. Compare:
1. Does ICP become more or less selective with more environments?
2. Does the precision/recall improve with more environments?
3. Why does more environments increase power?

**Expected:** With 6 environments, ICP should exclude all spurious features more confidently.

**Hints:**
- Use `generate_causal_dataset(n_per_env=200, n_envs=6)` — but you need to extend `env_hidden_strength` to length 6
- Alternatively, modify `generate_causal_dataset` to accept `env_hidden_strength` as a parameter

In [ ]:
# YOUR CODE HERE
# ──────────────────────────────────────────────────────────────────
# Step 1: Generate dataset with 6 environments
#         (extend env_hidden_strength to [2.0, -1.5, 1.0, -2.0, 1.8, -1.2])

# Step 2: Standardise the new dataset

# Step 3: Run greedy_icp() with alpha=0.05

# Step 4: Print selected features, precision, and recall

# Step 5: Explain: why does more environments increase ICP power?

In [ ]:
# AUTO-CHECK
def test_exercise_icp_envs():
    print("Exercise check:")
    print("  - With 4 environments: ICP should select {X1_causal, X2_causal}")
    print("  - With 6 environments: ICP should be equally or more precise")
    print("  - More environments → more power to detect invariance violations")
    print("  - Each additional environment adds another statistical test,")
    print("    making it harder for spurious features to pass all tests simultaneously")

test_exercise_icp_envs()

## 9. Summary

### Key Takeaways

1. **ICP exploits invariance:** The conditional distribution $P(Y \mid X_{S^*})$ is stable across environments for true causal parents $S^*$. ICP identifies features with this property.
2. **Environments must be genuine:** Time periods, experimental conditions, or distributional shifts. Random splits give no power.
3. **ICP is conservative:** It may miss weak causal features (low power) but rarely selects spurious ones (controlled false positive rate).
4. **Distribution shift test:** ICP-selected features degrade less when the distribution shifts to a new environment. Standard predictive features (Lasso, SHAP) degrade more.
5. **Greedy ICP** makes the procedure tractable for moderate $p$.

### What's Next

Notebook 03 provides a comprehensive comparison of all causal vs predictive selection methods under varying degrees of distribution shift, and develops the blended production workflow.

### References
- Peters, J., Bühlmann, P., & Meinshausen, N. (2016). Causal inference by using invariant prediction. *JRSS-B*, 78(5), 947–1012.
- Arjovsky, M., et al. (2019). Invariant risk minimization. *arXiv:1907.02893*.